# Invoice Automation AI Agent

## AI Legends 2026 — AI Agent Automation Track

Энэ notebook нь invoice зураг болон PDF файлуудаас мэдээлэл олборлож, master database-тай тулган шалгаж, эрсдэл илрүүлж, санхүүгийн ангилал оноож, эцсийн бизнес шийдвэр гаргадаг AI agent pipeline юм.

**Гол боломжууд:**

- PDF/JPG/PNG invoice файлыг автоматаар илрүүлэх
- Groq Vision model ашиглан structured field extraction хийх
- Vendor, bank account, amount, date, duplicate validation хийх
- `AUTO_POST`, `HUMAN_APPROVAL`, `DENY` шийдвэр гаргах
- Нийт үр дүн дээр chatbot-style Q&A хийх
- Optional Gradio interface ашиглан demo хийх

> Энэ notebook нь шинэ dataset нэмэгдсэн үед input folder-ийг hardcode хийхгүйгээр автоматаар scan хийхээр бүтээгдсэн.

## 1. Competition Requirement Mapping

| Competition requirement | Notebook section |
|---|---|
| Extract information from invoice image/PDF | Vision-based extraction |
| Classify invoice into financial category | Category classification |
| Detect errors and risks | Validation + risk flagging |
| Make final decision | Final decision logic |
| Answer aggregate questions | Chatbot Q&A agent |
| Public notebook reproducibility | Clear config, outputs, dependency list |

**Required risk types:**

- `AMOUNT_MISMATCH`
- `UNREGISTERED_VENDOR`
- `INVALID_DATE`
- `BANK_ACCOUNT_MISMATCH`
- `DUPLICATE`

**Required final decisions:**

- `AUTO_POST`
- `HUMAN_APPROVAL`
- `DENY`

## 2. Install Dependencies

Kaggle runtime дээр зарим library байхгүй байж болно. Энэ cell нь шаардлагатай package-уудыг суулгана.

In [2]:
!pip install -q groq pymupdf pillow pandas numpy rapidfuzz gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.9 MB/s eta 0:00:00:00:01


## 3. Import Libraries

In [53]:
import os
import re
import io
import json
import time
import base64
import sqlite3
import traceback
from pathlib import Path
from datetime import datetime, date
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from rapidfuzz import fuzz, process

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from groq import Groq
except Exception:
    Groq = None

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

print('Сангууд амжилттай import хийгдлээ.')

Сангууд амжилттай import хийгдлээ.


## 4. Configuration

Энэ хэсэгт input/output path, model name, processing limit зэргийг тохируулна.

In [54]:
# Kaggle competition dataset root. Шинэ dataset нэмэгдсэн ч энэ folder дотор scan хийнэ.
DEFAULT_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_NAME = 'ai-legends-2026-ai-agents-automation'
COMPETITION_DIR = DEFAULT_INPUT_ROOT / 'competitions' / COMPETITION_NAME

# Kaggle working output folder
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Vision model. Groq дээр боломжтой vision model ашиглана.
GROQ_VISION_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Debug үед эхний хэдэн invoice боловсруулах. None бол бүх invoice.
MAX_FILES_TO_PROCESS = None

# PDF-ийн эхний хэдэн page унших. Invoice ихэнхдээ 1 page байдаг.
MAX_PDF_PAGES = 2

print('Input root:', DEFAULT_INPUT_ROOT)
print('Competition dir:', COMPETITION_DIR)
print('Output dir:', OUTPUT_DIR)

Input root: /kaggle/input
Competition dir: /kaggle/input/competitions/ai-legends-2026-ai-agents-automation
Output dir: /kaggle/working


## 5. Load Multiple Groq API Keys

Kaggle Secrets дээр дараах нэрүүдээр key хадгалж болно:

- `GROQ_API_KEY_1`
- `GROQ_API_KEY_2`
- `GROQ_API_KEY_3`
- `GROQ_API_KEY_4`
- `GROQ_API_KEY_5`

Fallback байдлаар `GROQ_API_KEY` болон `API` нэрийг мөн шалгана. API key-г notebook дотор шууд бичихгүй.

In [71]:
# ============================================================
# 5. Groq API Key Setup
# ============================================================

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_SECRET_NAMES = [
    "groq_API_1_new",
    "groq_API_2_new",
    "groq_API_3_new",
    "groq_API_4_new",
    "groq_API_5_new",
]

groq_api_keys = []

for secret_name in GROQ_SECRET_NAMES:
    try:
        key = user_secrets.get_secret(secret_name)

        if key and str(key).strip():
            groq_api_keys.append(str(key).strip())
            print(f"Loaded secret: {secret_name}")
        else:
            print(f"Empty secret skipped: {secret_name}")

    except Exception as e:
        print(f"Could not load secret: {secret_name} | {e}")

if len(groq_api_keys) == 0:
    raise ValueError("No Groq API keys loaded. Please add Kaggle Secrets first.")

api_keys = groq_api_keys
GROQ_API_KEYS = groq_api_keys
GROQ_KEYS = groq_api_keys

print(f"Total Groq API keys loaded: {len(groq_api_keys)}")
print(f"API aliases ready: {len(api_keys)}")
print("First key preview:", groq_api_keys[0][:8] + "..." if groq_api_keys else "NO KEY")

Loaded secret: groq_API_1_new
Loaded secret: groq_API_2_new
Loaded secret: groq_API_3_new
Loaded secret: groq_API_4_new
Loaded secret: groq_API_5_new
Total Groq API keys loaded: 5
API aliases ready: 5
First key preview: gsk_Dc7c...


## 6. Auto-detect Dataset Files

Энэ хэсэг шинэ data ирсэн үед бүх PDF/JPG/PNG invoice болон database файлыг автоматаар олно.

In [56]:
def find_existing_root() -> Path:
    """Choose the most likely dataset root."""
    if COMPETITION_DIR.exists():
        return COMPETITION_DIR
    if DEFAULT_INPUT_ROOT.exists():
        return DEFAULT_INPUT_ROOT
    return Path('.')

DATA_ROOT = find_existing_root()
print('Selected DATA_ROOT:', DATA_ROOT)


def scan_invoice_files(root: Path) -> List[Path]:
    patterns = ['*.jpg', '*.jpeg', '*.png', '*.pdf']
    files = []
    for pattern in patterns:
        files.extend(root.rglob(pattern))
    # exclude output/generated folders if running locally
    files = [p for p in files if 'outputs' not in str(p).lower()]
    return sorted(files)


def scan_database_files(root: Path) -> List[Path]:
    patterns = ['*.db', '*.sqlite', '*.sqlite3', '*.csv', '*.xlsx']
    files = []
    for pattern in patterns:
        files.extend(root.rglob(pattern))
    return sorted(files)

invoice_files = scan_invoice_files(DATA_ROOT)
database_files = scan_database_files(DATA_ROOT)

if MAX_FILES_TO_PROCESS:
    invoice_files = invoice_files[:MAX_FILES_TO_PROCESS]

print(f'Олдсон invoice файлын тоо: {len(invoice_files)}')
print(f'Олдсон database/data файлын тоо: {len(database_files)}')

pd.DataFrame({'invoice_file': [str(p) for p in invoice_files[:20]]})

Selected DATA_ROOT: /kaggle/input/competitions/ai-legends-2026-ai-agents-automation
Олдсон invoice файлын тоо: 200
Олдсон database/data файлын тоо: 1


,invoice_file
0,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_001.pdf
1,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_002.png
2,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_003.pdf
3,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_004.pdf
4,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_005.pdf
5,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_006.png
6,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_007.png
7,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_008.pdf
8,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_009.png
9,/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_010.png


## 7. Load Master Database

Master database нь vendor, category, historical invoice зэрэг structured мэдээллийг агуулна. Энэ хэсэг SQLite database-г автоматаар уншиж, table бүрийг DataFrame болгоно.

In [57]:
def load_sqlite_database(db_path: Path) -> Dict[str, pd.DataFrame]:
    tables = {}
    try:
        conn = sqlite3.connect(db_path)
        table_names = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)['name'].tolist()
        for table in table_names:
            try:
                tables[table] = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
            except Exception as e:
                print(f'{table} унших үед алдаа: {e}')
        conn.close()
    except Exception as e:
        print('SQLite database унших үед алдаа:', e)
    return tables

sqlite_files = [p for p in database_files if p.suffix.lower() in ['.db', '.sqlite', '.sqlite3']]
MASTER_DB_PATH = sqlite_files[0] if sqlite_files else None

master_tables = {}
if MASTER_DB_PATH:
    print('Master database:', MASTER_DB_PATH)
    master_tables = load_sqlite_database(MASTER_DB_PATH)
    print('Tables:', list(master_tables.keys()))
else:
    print('Master SQLite database олдсонгүй.')

for name, df in master_tables.items():
    print(f'\n{name}: shape={df.shape}')
    display(df.head())

Master database: /kaggle/input/competitions/ai-legends-2026-ai-agents-automation/master_invoices_database.db
Tables: ['Vendors', 'sqlite_sequence', 'Items', 'InvoiceCategories', 'Invoices', 'InvoiceLines']

Vendors: shape=(10, 7)


,ID,Name,Bank,Account,Email,RegisteredDate,Status
0,1,Демо Компани-1,Демо Банк 1,5001122334,finance@demo1.mn,2021-01-24,active
1,2,Демо Компани-2,Демо Банк 1,1102003004,billing@demo2.mn,2023-11-16,active
2,3,Демо Компани-3,Демо Банк 1,4001122334,accounts@demo3.mn,2023-09-05,active
3,4,Демо Компани-4,Демо Банк 2,5007788990,invoicing@demo4.mn,2023-06-27,active
4,5,Демо Компани-5,Демо Банк 2,1009900880,info@demo5.mn,2023-01-16,active



sqlite_sequence: shape=(5, 2)


,name,seq
0,Vendors,10
1,Items,50
2,InvoiceCategories,10
3,Invoices,596
4,InvoiceLines,719



Items: shape=(50, 3)


...

,ID,Name,Description
0,1,Түрээсийн зардал,"Оффис, серверийн өрөө, агуулах, зогсоолын түрээс, форклифт түрээс"
1,2,Ашиглалтын зардал,"Цахилгаан, дулаан, ус, хог, цэвэрлэгээ, харуул хамгаалалт, агааржуулалт"
2,3,Мэдээллийн технологийн зардал,"Сервер, интернэт, лиценз, програм хангамж, кибер аюулгүй байдал, домэйн, SSL, нөөцлөлт, вэб байршуулалт, бараа бүртг..."
3,4,Тоног төхөөрөмж,"Монитор, принтер, IP камер, сүлжээний кабель угсралт"
4,5,"Тээвэр, логистик","Ачаа тээвэр, шатахуун, хүргэлт, ачаа ачих буулгах, гаалийн зардал, GPS трекер"



Invoices: shape=(596, 8)


,ID,VendorName,InvoiceDate,DueDate,GrandTotal,InvoiceCategoryID,Status,ApprovedDate
0,1,Демо Компани-10,2025-12-09,2025-12-24,280000,5,approved,2025-12-26
1,2,Демо Компани-3,2026-03-06,2026-03-21,1760000,4,approved,2026-03-25
2,3,Демо Компани-10,2025-05-15,2025-05-30,150000,9,approved,2025-06-06
3,4,Демо Компани-9,2025-04-28,2025-05-13,320000,7,approved,2025-05-20
4,5,Демо Компани-6,2025-10-03,2025-10-18,550000,2,approved,2025-10-23



InvoiceLines: shape=(719, 6)


,ID,InvoiceID,ItemID,Qty,UnitPrice,Total
0,1,1,38,1,280000,280000
1,2,2,10,4,280000,1120000
2,3,2,11,2,320000,640000
3,4,3,47,1,150000,150000
4,5,4,46,1,320000,320000


## 8. Master Data Helpers

Column нэр өөр байсан ч аль болох уян хатан ажиллах helper функцууд.

In [58]:
def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalize_number(value: Any) -> Optional[float]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value)
    text = text.replace(',', '').replace('₮', '').replace('mnt', '').replace('MNT', '')
    text = re.sub(r'[^0-9.\-]', '', text)
    if text in ['', '.', '-', '-.']:
        return None
    try:
        return float(text)
    except Exception:
        return None


def find_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        c_norm = c.lower().replace('_', '').replace(' ', '')
        for cand in candidates:
            cand_norm = cand.lower().replace('_', '').replace(' ', '')
            if cand_norm in c_norm or c_norm in cand_norm:
                return c
    return None


def get_table(possible_names: List[str]) -> Optional[pd.DataFrame]:
    lower_map = {name.lower(): name for name in master_tables.keys()}
    for name in possible_names:
        if name.lower() in lower_map:
            return master_tables[lower_map[name.lower()]]
    # fuzzy partial match
    for table_name, df in master_tables.items():
        for name in possible_names:
            if name.lower() in table_name.lower():
                return df
    return None

vendors_df = get_table(['Vendors', 'Vendor', 'Suppliers'])
items_df = get_table(['Items', 'Item'])
categories_df = get_table(['InvoiceCategories', 'Categories', 'Category'])
historical_invoices_df = get_table(['Invoices', 'Invoice', 'HistoricalInvoices'])

print('vendors_df:', None if vendors_df is None else vendors_df.shape)
print('items_df:', None if items_df is None else items_df.shape)
print('categories_df:', None if categories_df is None else categories_df.shape)
print('historical_invoices_df:', None if historical_invoices_df is None else historical_invoices_df.shape)

vendors_df: (10, 7)
items_df: (50, 3)
categories_df: (10, 3)
historical_invoices_df: (596, 8)


## 9. Image/PDF Conversion Helpers

PDF invoice-ийг image болгож Vision model-д дамжуулна.

In [59]:
def image_to_data_url(image: Image.Image, max_size: int = 1600) -> str:
    """Convert PIL image to base64 data URL."""
    image = image.convert('RGB')
    w, h = image.size
    scale = min(max_size / max(w, h), 1.0)
    if scale < 1.0:
        image = image.resize((int(w * scale), int(h * scale)))

    buffer = io.BytesIO()
    image.save(buffer, format='JPEG', quality=90)
    b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return f'data:image/jpeg;base64,{b64}'


def file_to_images(file_path: Path, max_pdf_pages: int = MAX_PDF_PAGES) -> List[Image.Image]:
    """Load image file or convert PDF pages to PIL images."""
    suffix = file_path.suffix.lower()
    images = []

    if suffix in ['.jpg', '.jpeg', '.png']:
        images.append(Image.open(file_path).convert('RGB'))
        return images

    if suffix == '.pdf':
        if fitz is None:
            raise RuntimeError('PyMuPDF байхгүй тул PDF унших боломжгүй.')
        doc = fitz.open(str(file_path))
        for page_index in range(min(len(doc), max_pdf_pages)):
            page = doc[page_index]
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
            img = Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')
            images.append(img)
        doc.close()
        return images

    raise ValueError(f'Unsupported file type: {suffix}')

# Test local preview only if files exist
if invoice_files:
    test_images = file_to_images(invoice_files[0])
    print(f'Preview file: {invoice_files[0].name}, image pages: {len(test_images)}, first size: {test_images[0].size}')
else:
    print('Invoice file олдсонгүй.')

Preview file: invoice_001.pdf, image pages: 1, first size: (1191, 1684)


## 10. Vision-based Invoice Extraction

Groq Vision model invoice-оос structured JSON талбаруудыг буцаана.

In [60]:
EXTRACTION_SYSTEM_PROMPT = """
You are an invoice information extraction agent.
Extract structured fields from Mongolian or English invoice images.
Return ONLY valid JSON. No markdown. No explanation.
If a value is missing, use null.
"""

EXTRACTION_USER_PROMPT = """
Extract the following invoice fields and return valid JSON only:

{
  "invoice_number": null,
  "vendor_name": null,
  "invoice_date": null,
  "due_date": null,
  "bank_name": null,
  "bank_account": null,
  "email": null,
  "currency": "MNT",
  "subtotal": null,
  "tax": null,
  "total_amount": null,
  "items": [
    {
      "description": null,
      "quantity": null,
      "unit_price": null,
      "line_total": null
    }
  ]
}

Rules:
- Keep vendor_name as written on the invoice.
- Convert amounts to numbers if possible.
- Dates should be ISO format YYYY-MM-DD if possible.
- For Mongolian invoices, preserve Mongolian names.
- Return JSON only.
"""


def extract_json_from_text(text: str) -> Dict[str, Any]:
    """Parse JSON object from model response."""
    if not text:
        return {}
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}
    return {}


def call_groq_with_fallback(messages: List[Dict[str, Any]], model: str = GROQ_VISION_MODEL, max_tokens: int = 1800) -> str:
    """Call Groq with multiple API key fallback."""
    if Groq is None:
        raise RuntimeError('groq package import хийгдээгүй байна.')
    if not GROQ_API_KEYS:
        raise RuntimeError('Groq API key олдсонгүй.')

    last_error = None
    for idx, api_key in enumerate(GROQ_API_KEYS, start=1):
        try:
            client = Groq(api_key=api_key, max_retries=1, timeout=60)
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                max_completion_tokens=max_tokens,
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f'Groq API key {idx} дээр алдаа гарлаа. Дараагийн key ашиглана...')
            time.sleep(2)
    raise RuntimeError(f'Бүх Groq API key амжилтгүй боллоо. Last error: {last_error}')


def extract_invoice_with_groq(file_path: Path) -> Dict[str, Any]:
    """Extract invoice fields using Groq Vision."""
    images = file_to_images(file_path)
    content = [{"type": "text", "text": EXTRACTION_USER_PROMPT}]
    for img in images:
        content.append({"type": "image_url", "image_url": {"url": image_to_data_url(img)}})

    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]
    raw = call_groq_with_fallback(messages)
    parsed = extract_json_from_text(raw)
    parsed['_raw_model_response'] = raw
    return parsed


def fallback_empty_extraction(file_path: Path) -> Dict[str, Any]:
    """Fallback when API is not available. Keeps pipeline running but marks extraction as failed."""
    return {
        'invoice_number': file_path.stem,
        'vendor_name': None,
        'invoice_date': None,
        'due_date': None,
        'bank_name': None,
        'bank_account': None,
        'email': None,
        'currency': 'MNT',
        'subtotal': None,
        'tax': None,
        'total_amount': None,
        'items': [],
        '_raw_model_response': None,
        '_fallback_mode': True,
    }

print('Vision extraction functions ready.')

Vision extraction functions ready.


## 11. Field Normalization

Model output-ийг validation хийхэд тохиромжтой structured format болгоно.

In [61]:
def normalize_date(value: Any) -> Optional[str]:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None

    # common separators and Mongolian date markers
    text = text.replace('он', '-').replace('сар', '-').replace('өдөр', '')
    text = re.sub(r'[./]', '-', text)
    text = re.sub(r'\s+', '', text)

    candidates = [text]
    # Extract YYYY-MM-DD-like pattern
    m = re.search(r'(20\d{2}|19\d{2})[-年]?(\d{1,2})[-月]?(\d{1,2})', text)
    if m:
        candidates.insert(0, f'{m.group(1)}-{m.group(2)}-{m.group(3)}')

    for cand in candidates:
        for fmt in ['%Y-%m-%d', '%Y-%m-%d', '%d-%m-%Y', '%m-%d-%Y']:
            try:
                dt = datetime.strptime(cand, fmt).date()
                return dt.isoformat()
            except Exception:
                pass
    return None


def normalize_items(items: Any) -> List[Dict[str, Any]]:
    if not isinstance(items, list):
        return []
    normalized = []
    for item in items:
        if not isinstance(item, dict):
            continue
        q = normalize_number(item.get('quantity'))
        u = normalize_number(item.get('unit_price'))
        lt = normalize_number(item.get('line_total'))
        normalized.append({
            'description': item.get('description'),
            'quantity': q,
            'unit_price': u,
            'line_total': lt,
            'calculated_line_total': q * u if q is not None and u is not None else None,
        })
    return normalized


def normalize_invoice_data(raw: Dict[str, Any], file_path: Path) -> Dict[str, Any]:
    items = normalize_items(raw.get('items'))
    total_from_items = sum([i['calculated_line_total'] for i in items if i.get('calculated_line_total') is not None])
    total_from_items = total_from_items if total_from_items > 0 else None

    data = {
        'file_name': file_path.name,
        'file_path': str(file_path),
        'file_type': file_path.suffix.lower().replace('.', ''),
        'invoice_number': raw.get('invoice_number') or file_path.stem,
        'vendor_name': raw.get('vendor_name'),
        'invoice_date': normalize_date(raw.get('invoice_date')),
        'due_date': normalize_date(raw.get('due_date')),
        'bank_name': raw.get('bank_name'),
        'bank_account': raw.get('bank_account'),
        'email': raw.get('email'),
        'currency': raw.get('currency') or 'MNT',
        'subtotal': normalize_number(raw.get('subtotal')),
        'tax': normalize_number(raw.get('tax')),
        'total_amount': normalize_number(raw.get('total_amount')),
        'items': items,
        'items_text': ' | '.join([str(i.get('description') or '') for i in items]),
        'calculated_total_from_items': total_from_items,
        'raw_model_response': raw.get('_raw_model_response'),
        'fallback_mode': bool(raw.get('_fallback_mode', False)),
    }
    return data

print('Normalization functions ready.')

Normalization functions ready.


## 12. Category Classification

Category-г historical database болон keyword rule ашиглан онооно. Custom ML training хийхгүй, учир нь dataset өөрчлөгдөх боломжтой бөгөөд rule + historical matching нь илүү тайлбарлагдахуйц.

In [62]:
CATEGORY_KEYWORDS = {
    'Software / IT': ['сервер', 'server', 'hosting', 'ssl', 'software', 'domain', 'cloud', 'api', 'license', 'лиценз'],
    'Office Supplies': ['оффис', 'бичиг', 'цаас', 'printer', 'paper', 'supplies', 'хэрэгсэл'],
    'Utilities': ['цахилгаан', 'ус', 'дулаан', 'internet', 'интернет', 'utility', 'холбоо'],
    'Travel Expense': ['томилолт', 'зочид', 'hotel', 'taxi', 'такси', 'flight', 'нислэг', 'travel'],
    'Maintenance': ['засвар', 'үйлчилгээ', 'maintenance', 'repair', 'support'],
    'Training / Consulting': ['сургалт', 'зөвлөгөө', 'consulting', 'training', 'workshop'],
}


def historical_category_match(vendor_name: Any) -> Optional[str]:
    if historical_invoices_df is None or vendor_name is None:
        return None
    vendor_col = find_column(historical_invoices_df, ['VendorName', 'Vendor', 'vendor_name', 'Name'])
    category_col = find_column(historical_invoices_df, ['Category', 'InvoiceCategory', 'category_name'])
    if not vendor_col or not category_col:
        return None

    target = normalize_text(vendor_name)
    if not target:
        return None
    temp = historical_invoices_df.copy()
    temp['_score'] = temp[vendor_col].apply(lambda x: fuzz.token_sort_ratio(target, normalize_text(x)))
    best = temp.sort_values('_score', ascending=False).head(1)
    if not best.empty and best['_score'].iloc[0] >= 85:
        return str(best[category_col].iloc[0])
    return None


def keyword_category_match(text: str) -> str:
    text_norm = normalize_text(text)
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(k.lower() in text_norm for k in keywords):
            return category
    return 'Other'


def classify_category(invoice: Dict[str, Any]) -> str:
    hist = historical_category_match(invoice.get('vendor_name'))
    if hist:
        return hist
    combined_text = ' '.join([
        str(invoice.get('vendor_name') or ''),
        str(invoice.get('items_text') or ''),
    ])
    return keyword_category_match(combined_text)

print('Category classification ready.')

Category classification ready.


## 13. Validation Rules

Энэ хэсэг competition-д шаардсан гол зөрчлүүдийг илрүүлнэ.

In [63]:
def validate_registered_vendor(vendor_name: Any) -> Tuple[bool, Optional[str], float]:
    if vendors_df is None or vendor_name is None:
        return False, None, 0.0
    name_col = find_column(vendors_df, ['Name', 'VendorName', 'Vendor', 'vendor_name'])
    if not name_col:
        return False, None, 0.0

    choices = vendors_df[name_col].dropna().astype(str).tolist()
    if not choices:
        return False, None, 0.0

    match = process.extractOne(str(vendor_name), choices, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 80:
        return True, match[0], float(match[1])
    return False, match[0] if match else None, float(match[1]) if match else 0.0


def validate_amount(invoice: Dict[str, Any], tolerance: float = 1.0) -> bool:
    total = invoice.get('total_amount')
    calculated = invoice.get('calculated_total_from_items')
    if total is None or calculated is None:
        # Missing fields are handled separately; do not mark as mismatch by default.
        return True
    return abs(float(total) - float(calculated)) <= tolerance


def validate_date(invoice: Dict[str, Any]) -> bool:
    invoice_date = invoice.get('invoice_date')
    due_date = invoice.get('due_date')
    today = date.today()

    try:
        inv_dt = datetime.fromisoformat(invoice_date).date() if invoice_date else None
        due_dt = datetime.fromisoformat(due_date).date() if due_date else None
    except Exception:
        return False

    if inv_dt and inv_dt > today:
        return False
    if inv_dt and due_dt and due_dt < inv_dt:
        return False
    return True


def validate_bank_account(invoice: Dict[str, Any], matched_vendor_name: Optional[str]) -> bool:
    """Check bank account if Vendors table has an account-like column. If unavailable, return True."""
    if vendors_df is None or not matched_vendor_name:
        return True

    vendor_col = find_column(vendors_df, ['Name', 'VendorName', 'Vendor', 'vendor_name'])
    account_col = find_column(vendors_df, ['BankAccount', 'AccountNumber', 'Account', 'bank_account', 'Данс'])
    if not vendor_col or not account_col:
        return True

    invoice_account = re.sub(r'\D', '', str(invoice.get('bank_account') or ''))
    if not invoice_account:
        return True

    row = vendors_df[vendors_df[vendor_col].astype(str) == str(matched_vendor_name)]
    if row.empty:
        return True
    master_account = re.sub(r'\D', '', str(row.iloc[0][account_col]))
    if not master_account:
        return True
    return invoice_account == master_account


def detect_duplicate(invoice: Dict[str, Any]) -> bool:
    if historical_invoices_df is None:
        return False

    inv_num_col = find_column(historical_invoices_df, ['InvoiceNumber', 'invoice_number', 'Number', 'ID'])
    vendor_col = find_column(historical_invoices_df, ['VendorName', 'Vendor', 'vendor_name', 'Name'])
    total_col = find_column(historical_invoices_df, ['GrandTotal', 'TotalAmount', 'total_amount', 'Amount'])
    date_col = find_column(historical_invoices_df, ['InvoiceDate', 'Date', 'invoice_date'])

    inv_num = normalize_text(invoice.get('invoice_number'))
    if inv_num and inv_num_col:
        if historical_invoices_df[inv_num_col].astype(str).apply(normalize_text).eq(inv_num).any():
            return True

    # fallback duplicate: same vendor + same amount + same date
    if vendor_col and total_col:
        vendor = normalize_text(invoice.get('vendor_name'))
        total = invoice.get('total_amount')
        inv_date = invoice.get('invoice_date')
        for _, row in historical_invoices_df.iterrows():
            vendor_score = fuzz.token_sort_ratio(vendor, normalize_text(row.get(vendor_col))) if vendor else 0
            total_same = total is not None and normalize_number(row.get(total_col)) is not None and abs(total - normalize_number(row.get(total_col))) <= 1
            date_same = True
            if date_col and inv_date:
                date_same = normalize_date(row.get(date_col)) == inv_date
            if vendor_score >= 85 and total_same and date_same:
                return True
    return False

print('Validation rules ready.')

Validation rules ready.


## 14. Risk Flagging and Final Decision Logic

Business decision rule:

- Serious risk → `DENY`
- New/uncertain invoice → `HUMAN_APPROVAL`
- Clean and known invoice → `AUTO_POST`

In [64]:
def assign_risk_flags(invoice: Dict[str, Any]) -> Dict[str, Any]:
    flags = []

    registered, matched_vendor, vendor_score = validate_registered_vendor(invoice.get('vendor_name'))
    amount_ok = validate_amount(invoice)
    date_ok = validate_date(invoice)
    bank_ok = validate_bank_account(invoice, matched_vendor)
    duplicate = detect_duplicate(invoice)

    if not amount_ok:
        flags.append('AMOUNT_MISMATCH')
    if not registered:
        flags.append('UNREGISTERED_VENDOR')
    if not date_ok:
        flags.append('INVALID_DATE')
    if not bank_ok:
        flags.append('BANK_ACCOUNT_MISMATCH')
    if duplicate:
        flags.append('DUPLICATE')
    if invoice.get('fallback_mode'):
        flags.append('LOW_CONFIDENCE_EXTRACTION')

    invoice.update({
        'is_registered_vendor': registered,
        'matched_vendor_name': matched_vendor,
        'vendor_match_score': vendor_score,
        'amount_check': amount_ok,
        'date_check': date_ok,
        'bank_account_check': bank_ok,
        'duplicate_check': duplicate,
        'risk_flags': flags,
    })
    return invoice


def make_final_decision(invoice: Dict[str, Any]) -> str:
    flags = set(invoice.get('risk_flags') or [])
    deny_flags = {'AMOUNT_MISMATCH', 'INVALID_DATE', 'BANK_ACCOUNT_MISMATCH', 'DUPLICATE'}

    if flags.intersection(deny_flags):
        return 'DENY'
    if 'UNREGISTERED_VENDOR' in flags or 'LOW_CONFIDENCE_EXTRACTION' in flags:
        return 'HUMAN_APPROVAL'
    return 'AUTO_POST'


def make_explanation(invoice: Dict[str, Any]) -> str:
    flags = invoice.get('risk_flags') or []
    if not flags:
        return 'No risk detected. Vendor, date, amount and duplicate checks passed.'
    return 'Detected risk flags: ' + ', '.join(flags)

print('Risk flagging and decision logic ready.')

Risk flagging and decision logic ready.


## 15. Single Invoice Processing Function

In [65]:
def process_single_invoice(file_path: Path) -> Dict[str, Any]:
    start = time.time()
    result = {
        'file_name': file_path.name,
        'file_path': str(file_path),
        'file_type': file_path.suffix.lower().replace('.', ''),
        'processing_status': 'FAILED',
        'error_message': None,
    }

    try:
        if GROQ_API_KEYS:
            raw = extract_invoice_with_groq(file_path)
        else:
            raw = fallback_empty_extraction(file_path)

        invoice = normalize_invoice_data(raw, file_path)
        invoice['category'] = classify_category(invoice)
        invoice = assign_risk_flags(invoice)
        invoice['final_decision'] = make_final_decision(invoice)
        invoice['explanation'] = make_explanation(invoice)
        invoice['processing_status'] = 'SUCCESS'
        invoice['processing_time_sec'] = round(time.time() - start, 2)
        return invoice

    except Exception as e:
        result['error_message'] = str(e)
        result['processing_time_sec'] = round(time.time() - start, 2)
        return result

print('Single invoice processor ready.')

Single invoice processor ready.


> checking all files

In [67]:
print("Нийт олдсон invoice файл:", len(invoice_files))
print("="*40)

for f in invoice_files[:10]:
    print(f)
print("="*40)
print("Нийт invoice_files:", len(invoice_files))
print("Unique invoice_files:", len(set(invoice_files)))

if len(invoice_files) != len(set(invoice_files)):
    print("Анхаар: duplicate file path байна")
else:
    print("Duplicate file path байхгүй")

print("="*40)

for i, file_path in enumerate(invoice_files, start=1):
    print(f"[{i}/{len(invoice_files)}] Processing: {file_path}")

Нийт олдсон invoice файл: 200
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_001.pdf
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_002.png
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_003.pdf
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_004.pdf
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_005.pdf
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_006.png
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_007.png
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_008.pdf
/kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_009.png
/kaggle/input/competitions/ai-legends-2026-ai-agents

## 16. Хурдан ба найдвартай batch processing

Энэ хэсэгт invoice файлуудыг нэг нэгээр нь тогтвортой байдлаар боловсруулна.  
Гол зорилго нь Kaggle орчинд pipeline-ийг найдвартай, дахин ажиллуулах боломжтой байлгах юм.

Runtime-ийг багасгахын тулд дараах аргуудыг ашигласан:

- **Ухаалаг input сонголт**: боломжтой үед final evaluation invoice folder-ийг түрүүлж сонгоно.
- **Quick test mode**: хөгжүүлэлтийн үед цөөн invoice дээр хурдан шалгана.
- **Cache**: өмнө боловсруулсан invoice-ийн үр дүнг хадгалж, дахин API call хийхээс сэргийлнэ.
- **Failure handling**: алдаатай invoice-ийг final result-оос хасахгүй, `HUMAN_APPROVAL` гэж тэмдэглэнэ.
- **Progress logging**: одоогийн файл, status, decision, дундаж хугацаа, үлдсэн хугацааны тооцоог харуулна.

Final run хийх үед `QUICK_TEST = False` болгож бүх selected invoice-ийг боловсруулна.

* chache clear optional

In [74]:
import shutil
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/invoice_cache")

if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)

CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Cache cleared:", CACHE_DIR)

Cache cleared: /kaggle/working/invoice_cache


In [85]:
# ============================================================
# 16. Fast + Safe Batch Processing
# ============================================================

from pathlib import Path
import json
import time
import hashlib
import traceback
from datetime import datetime

# -----------------------------
# Batch configuration
# -----------------------------

QUICK_TEST = False          # Туршилт хийх үед True, final run хийх үед False
PROCESS_LIMIT = 10         # QUICK_TEST=True үед хэдэн invoice process хийх вэ
USE_CACHE = True           # Өмнө боловсруулсан result байвал дахин API call хийхгүй
FORCE_REPROCESS = False    # True бол cache үл тооно
PREFERRED_FOLDER_KEYWORD = "eval_agent/eval_agent"

CACHE_DIR = Path("/kaggle/working/invoice_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("========== BATCH CONFIG ==========")
print("QUICK_TEST      :", QUICK_TEST)
print("PROCESS_LIMIT   :", PROCESS_LIMIT)
print("USE_CACHE       :", USE_CACHE)
print("FORCE_REPROCESS :", FORCE_REPROCESS)
print("CACHE_DIR       :", CACHE_DIR)
print("==================================")


# -----------------------------
# Helper: cache path
# -----------------------------

def safe_cache_name(file_path: Path) -> str:
    """
    File path дээр үндэслэж unique cache filename үүсгэнэ.
    invoice_001.pdf гэх мэт filename давхцаж болох тул full path hash ашиглаж байна.
    """
    path_str = str(file_path)
    short_hash = hashlib.md5(path_str.encode("utf-8")).hexdigest()[:10]
    return f"{file_path.stem}_{short_hash}.json"


def get_cache_path(file_path: Path) -> Path:
    return CACHE_DIR / safe_cache_name(file_path)


def save_json_cache(file_path: Path, data: dict):
    cache_path = get_cache_path(file_path)
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json_cache(file_path: Path):
    cache_path = get_cache_path(file_path)
    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


# -----------------------------
# Helper: smart target selection
# -----------------------------

def choose_processing_files(all_invoice_files):
    """
    all_invoice_files дотроос processing target сонгоно.
    Priority:
    1. eval_agent/eval_agent folder байвал түүнийг сонгоно
    2. Байхгүй бол бүх invoice file-ийг unique байдлаар авна
    """
    all_invoice_files = [Path(p) for p in all_invoice_files]
    all_invoice_files = sorted(list(set(all_invoice_files)), key=lambda p: str(p))

    preferred_files = [
        p for p in all_invoice_files
        if PREFERRED_FOLDER_KEYWORD in str(p).replace("\\", "/")
    ]

    if len(preferred_files) > 0:
        selected = preferred_files
        selected_source = f"Preferred folder: {PREFERRED_FOLDER_KEYWORD}"
    else:
        selected = all_invoice_files
        selected_source = "Fallback: all detected invoice files"

    selected = sorted(list(set(selected)), key=lambda p: str(p))

    if QUICK_TEST:
        selected = selected[:PROCESS_LIMIT]

    return selected, selected_source


# -----------------------------
# Select files for processing
# -----------------------------

processing_files, selected_source = choose_processing_files(invoice_files)

print("\n========== INPUT SELECTION ==========")
print("Detected invoice files :", len(invoice_files))
print("Unique detected files  :", len(set(map(str, invoice_files))))
print("Selected source        :", selected_source)
print("Processing files       :", len(processing_files))
print("First file             :", processing_files[0] if processing_files else "None")
print("Last file              :", processing_files[-1] if processing_files else "None")
print("=====================================")

if len(processing_files) == 0:
    raise ValueError("No invoice files selected for processing.")


# -----------------------------
# Batch processing loop
# -----------------------------

processed_results = []
failed_files = []

start_all = time.time()

print("\n========== BATCH START ==========")

for idx, file_path in enumerate(processing_files, start=1):
    file_path = Path(file_path)
    one_start = time.time()

    print(f"\n[{idx}/{len(processing_files)}] Processing: {file_path.name}")

    try:
        cached = None

        if USE_CACHE and not FORCE_REPROCESS:
            cached = load_json_cache(file_path)

        if cached is not None:
            result = cached
            loaded_from_cache = True
            print("  → Cache ашиглав")
        else:
            result = process_single_invoice(file_path)
            loaded_from_cache = False

        # -----------------------------
        # Standardize result from API or cache
        # -----------------------------

        if not isinstance(result, dict):
            result = {
                "file_name": file_path.name,
                "file_path": str(file_path),
                "file_type": file_path.suffix.lower().replace(".", ""),
                "extraction_status": "FAILED",
                "final_decision": "HUMAN_APPROVAL",
                "needs_human_approval": True,
                "risk_flags": "LOW_CONFIDENCE_EXTRACTION",
                "error_types": "LOW_CONFIDENCE_EXTRACTION",
                "failure_reason": "process_single_invoice returned non-dict result.",
                "processed_at": datetime.now().isoformat(),
            }

        result.setdefault("file_name", file_path.name)
        result.setdefault("file_path", str(file_path))
        result.setdefault("file_type", file_path.suffix.lower().replace(".", ""))
        result.setdefault("extraction_status", "SUCCESS")
        result.setdefault("processed_at", datetime.now().isoformat())

        if isinstance(result.get("risk_flags"), list):
            result["risk_flags"] = ";".join(result["risk_flags"]) if result["risk_flags"] else "NONE"

        if result.get("risk_flags") in [None, "", [], "[]"]:
            result["risk_flags"] = "NONE"

        result.setdefault("error_types", result.get("risk_flags", "NONE"))

        result.setdefault(
            "needs_human_approval",
            result.get("final_decision") == "HUMAN_APPROVAL"
        )

        result["loaded_from_cache"] = loaded_from_cache

        # -----------------------------
        # Guard incomplete / unknown extraction
        # -----------------------------

        decision_text = str(result.get("final_decision", "")).strip().upper()
        vendor_text = str(result.get("vendor_name", "")).strip().upper()
        risk_text = str(result.get("risk_flags", "")).strip().upper()
        status_text = str(result.get("extraction_status", "")).strip().upper()

        incomplete_decision = decision_text in ["", "UNKNOWN", "NONE", "NAN", "NULL"]
        incomplete_vendor = vendor_text in ["", "UNKNOWN", "NONE", "NAN", "NULL"]
        failed_status = status_text in ["FAILED", "ERROR"]

        if incomplete_decision or incomplete_vendor or failed_status:
            result["extraction_status"] = "FAILED"
            result["final_decision"] = "HUMAN_APPROVAL"
            result["needs_human_approval"] = True

            if risk_text in ["", "NONE", "[]", "NAN", "NULL"]:
                result["risk_flags"] = "LOW_CONFIDENCE_EXTRACTION"
            elif "LOW_CONFIDENCE_EXTRACTION" not in risk_text:
                result["risk_flags"] = str(result["risk_flags"]) + ";LOW_CONFIDENCE_EXTRACTION"

            result["error_types"] = result["risk_flags"]
            result["failure_reason"] = (
                "Incomplete extraction or unknown decision. Sent to human approval."
            )

        if USE_CACHE:
            save_json_cache(file_path, result)
            print("  → Cache хадгалав")

        processed_results.append(result)

        elapsed_one = time.time() - one_start
        print(f"  → Done in {elapsed_one:.2f}s")
        print(f"  → Status: {result.get('extraction_status', 'UNKNOWN')}")
        print(f"  → Decision: {result.get('final_decision', 'UNKNOWN')}")

    except Exception as e:
        error_text = str(e)
        traceback_text = traceback.format_exc()

        failed_result = {
            "file_name": file_path.name,
            "file_path": str(file_path),
            "file_type": file_path.suffix.lower().replace(".", ""),
            "extraction_status": "FAILED",
            "final_decision": "HUMAN_APPROVAL",
            "needs_human_approval": True,
            "risk_flags": "EXTRACTION_FAILED",
            "error_types": "EXTRACTION_FAILED",
            "denial_reason": "",
            "failure_reason": error_text,
            "traceback": traceback_text,
            "processed_at": datetime.now().isoformat(),
            "loaded_from_cache": False,
        }

        failed_files.append(failed_result)
        processed_results.append(failed_result)

        if USE_CACHE:
            save_json_cache(file_path, failed_result)

        print("  → FAILED")
        print("  → Reason:", error_text)

    # Progress summary
    done_count = idx
    elapsed_total = time.time() - start_all
    avg_time = elapsed_total / done_count
    remaining = len(processing_files) - done_count
    estimated_remaining = avg_time * remaining

    print(
        f"  → Progress: {done_count}/{len(processing_files)} | "
        f"Avg: {avg_time:.2f}s/file | "
        f"ETA: {estimated_remaining/60:.1f} min"
    )

print("\n========== BATCH FINISHED ==========")

total_elapsed = time.time() - start_all

print("Total selected files :", len(processing_files))
print("Processed rows       :", len(processed_results))
failed_count_runtime = sum(
    1 for r in processed_results
    if str(r.get("extraction_status", "")).upper() == "FAILED"
)

print("Failed rows          :", failed_count_runtime)
print("Total time           :", f"{total_elapsed/60:.2f} min")

if len(processed_results) == len(processing_files):
    print("OK: бүх selected invoice result-д орсон байна")
else:
    print("ERROR: selected invoice болон result count зөрж байна")

# Main dataframe
results_df = pd.DataFrame(processed_results)

print("\n========== RESULT PREVIEW ==========")
print("results_df shape:", results_df.shape)

display_cols = [
    "file_name",
    "file_type",
    "extraction_status",
    "vendor_name",
    "category",
    "final_decision",
    "risk_flags",
    "loaded_from_cache",
]

existing_display_cols = [c for c in display_cols if c in results_df.columns]
display(results_df[existing_display_cols].head(10))

========== BATCH CONFIG ==========
QUICK_TEST      : False
PROCESS_LIMIT   : 10
USE_CACHE       : True
FORCE_REPROCESS : False
CACHE_DIR       : /kaggle/working/invoice_cache

========== INPUT SELECTION ==========
Detected invoice files : 200
Unique detected files  : 200
Selected source        : Preferred folder: eval_agent/eval_agent
Processing files       : 100
First file             : /kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_001.pdf
Last file              : /kaggle/input/competitions/ai-legends-2026-ai-agents-automation/eval_agent/eval_agent/invoice_100.pdf

========== BATCH START ==========

[1/100] Processing: invoice_001.pdf
  → Cache ашиглав
  → Cache хадгалав
  → Done in 0.00s
  → Status: SUCCESS
  → Decision: DENY
  → Progress: 1/100 | Avg: 0.00s/file | ETA: 0.0 min

[2/100] Processing: invoice_002.png
  → Cache ашиглав
  → Cache хадгалав
  → Done in 0.00s
  → Status: SUCCESS
  → Decision: AUTO_POST
  → Progress: 2/100 | Avg

,file_name,file_type,extraction_status,vendor_name,category,final_decision,risk_flags,loaded_from_cache
0,invoice_001.pdf,pdf,SUCCESS,Демо Компани-5,7,DENY,AMOUNT_MISMATCH,True
1,invoice_002.png,png,SUCCESS,Демо Компани-8,5,AUTO_POST,NONE,True
2,invoice_003.pdf,pdf,SUCCESS,Демо Компани-10,5,AUTO_POST,NONE,True
3,invoice_004.pdf,pdf,SUCCESS,Демо Компани-1,8,AUTO_POST,NONE,True
4,invoice_005.pdf,pdf,SUCCESS,Демо Компани-4,10,AUTO_POST,NONE,True
5,invoice_006.png,png,SUCCESS,Демо Компани-1,8,AUTO_POST,NONE,True
6,invoice_007.png,png,SUCCESS,Демо Компани-5,7,AUTO_POST,NONE,True
7,invoice_008.pdf,pdf,SUCCESS,Демо Компани-1,8,DENY,BANK_ACCOUNT_MISMATCH,True
8,invoice_009.png,png,SUCCESS,Демо Компани-2,10,AUTO_POST,NONE,True
9,invoice_010.png,png,SUCCESS,Демо Компани-1,8,DENY,BANK_ACCOUNT_MISMATCH,True


## 17. Result Consolidation and Output Export

This section converts the batch processing results into final structured outputs.

The goal is to make the result layer consistent, judge-friendly, and easy to verify.  
All processed invoices, including failed invoices, are kept in the final result table.

This section creates:

- `all_results.csv` — all processed invoices
- `clean_invoices.csv` — invoices with `AUTO_POST`
- `suspicious_invoices.csv` — invoices requiring `HUMAN_APPROVAL` or `DENY`
- `failed_files.csv` — invoices where extraction failed
- aggregate summary counts for evaluation questions

The final outputs are saved to `/kaggle/working/`.

In [87]:
# ============================================================
# 17. Result Consolidation and Output Export
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("========== RESULT CONSOLIDATION START ==========")

# ------------------------------------------------------------
# 1. Validate batch result
# ------------------------------------------------------------

if "results_df" not in globals():
    raise ValueError("results_df олдсонгүй. Эхлээд Section 16 Batch Processing ажиллуулна уу.")

if results_df.empty:
    raise ValueError("results_df хоосон байна. Batch processing үр дүн гаргаагүй байна.")

final_results_df = results_df.copy()

print("Initial results_df shape:", final_results_df.shape)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def normalize_text_value(value, default=""):
    if value is None:
        return default

    if isinstance(value, float) and pd.isna(value):
        return default

    text = str(value).strip()

    if text.lower() in ["nan", "none", "null", ""]:
        return default

    return text


def normalize_bool_value(value, default=False):
    if value is None:
        return default

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return bool(value)

    text = str(value).strip().lower()

    if text in ["true", "yes", "1", "registered", "match", "valid", "correct"]:
        return True

    if text in ["false", "no", "0", "unregistered", "mismatch", "invalid", "incorrect"]:
        return False

    return default


def normalize_risk_flags(value):
    """
    risk_flags баганыг CSV-friendly string format болгоно.
    Жишээ:
    [] -> NONE
    ["AMOUNT_MISMATCH"] -> AMOUNT_MISMATCH
    ["A", "B"] -> A;B
    """
    if value is None:
        return "NONE"

    if isinstance(value, float) and pd.isna(value):
        return "NONE"

    if isinstance(value, list):
        return ";".join([str(v).strip() for v in value if str(v).strip()]) or "NONE"

    text = str(value).strip()

    if text in ["", "[]", "None", "nan", "NaN", "null"]:
        return "NONE"

    # Remove simple list-like brackets if they exist
    text = text.replace("[", "").replace("]", "").replace("'", "").replace('"', "").strip()

    if text == "":
        return "NONE"

    # Convert comma-separated values to semicolon-separated
    text = ";".join([part.strip() for part in text.split(",") if part.strip()])

    return text if text else "NONE"


def has_flag(flag_text, target_flag):
    flag_text = normalize_risk_flags(flag_text)
    if flag_text == "NONE":
        return False

    flags = [f.strip().upper() for f in flag_text.split(";")]
    return target_flag.upper() in flags


def ensure_column(df, column_name, default_value):
    if column_name not in df.columns:
        df[column_name] = default_value
    return df


# ------------------------------------------------------------
# 3. Ensure required columns exist
# ------------------------------------------------------------

required_defaults = {
    "file_name": "",
    "file_path": "",
    "file_type": "",
    "vendor_name": "",
    "invoice_date": "",
    "due_date": "",
    "category": "",
    "bank_name": "",
    "account_number": "",
    "quantity": None,
    "unit_price": None,
    "grand_total": None,
    "calculated_total": None,
    "extraction_status": "SUCCESS",
    "final_decision": "HUMAN_APPROVAL",
    "risk_flags": "NONE",
    "error_types": "NONE",
    "denial_reason": "",
    "failure_reason": "",
    "vendor_registered": False,
    "bank_account_match": False,
    "math_correct": False,
    "date_valid": False,
    "is_duplicate": False,
    "needs_human_approval": False,
    "loaded_from_cache": False,
    "processed_at": datetime.now().isoformat(),
}

for col, default in required_defaults.items():
    final_results_df = ensure_column(final_results_df, col, default)


# ------------------------------------------------------------
# 4. Normalize core columns
# ------------------------------------------------------------

text_columns = [
    "file_name",
    "file_path",
    "file_type",
    "vendor_name",
    "invoice_date",
    "due_date",
    "category",
    "bank_name",
    "account_number",
    "extraction_status",
    "final_decision",
    "error_types",
    "denial_reason",
    "failure_reason",
    "processed_at",
]

for col in text_columns:
    final_results_df[col] = final_results_df[col].apply(lambda x: normalize_text_value(x, ""))

final_results_df["risk_flags"] = final_results_df["risk_flags"].apply(normalize_risk_flags)

bool_columns = [
    "vendor_registered",
    "bank_account_match",
    "math_correct",
    "date_valid",
    "is_duplicate",
    "needs_human_approval",
    "loaded_from_cache",
]

for col in bool_columns:
    final_results_df[col] = final_results_df[col].apply(lambda x: normalize_bool_value(x, False))


# ------------------------------------------------------------
# 5. Recalculate final decision from risk flags
# ------------------------------------------------------------

def decide_from_risk_flags(risk_flags, extraction_status="SUCCESS"):
    flags = normalize_risk_flags(risk_flags)

    status = normalize_text_value(extraction_status, "SUCCESS").upper()

    if status == "FAILED":
        return "HUMAN_APPROVAL"

    if flags == "NONE":
        return "AUTO_POST"

    flag_set = set([
        f.strip().upper()
        for f in flags.split(";")
        if f.strip()
    ])

    human_approval_flags = {
        "LOW_CONFIDENCE_EXTRACTION",
        "EXTRACTION_FAILED",
        "UNREGISTERED_VENDOR",
        "AMOUNT_MISMATCH",
    }

    deny_flags = {
        "BANK_ACCOUNT_MISMATCH",
        "DUPLICATE",
    }

    if flag_set & deny_flags:
        return "DENY"

    if flag_set & human_approval_flags:
        return "HUMAN_APPROVAL"

    return "HUMAN_APPROVAL"


final_results_df["final_decision"] = final_results_df.apply(
    lambda row: decide_from_risk_flags(
        row.get("risk_flags", "NONE"),
        row.get("extraction_status", "SUCCESS")
    ),
    axis=1
)
# Keep human approval flag consistent with recalculated final decision
final_results_df["needs_human_approval"] = final_results_df["final_decision"].apply(
    lambda x: x == "HUMAN_APPROVAL"
)
# ------------------------------------------------------------
# 6. Derive evaluation-friendly flags
# ------------------------------------------------------------

final_results_df["has_amount_mismatch"] = final_results_df["risk_flags"].apply(
    lambda x: has_flag(x, "AMOUNT_MISMATCH")
)

final_results_df["has_unregistered_vendor"] = final_results_df["risk_flags"].apply(
    lambda x: has_flag(x, "UNREGISTERED_VENDOR")
)

final_results_df["has_invalid_date"] = final_results_df["risk_flags"].apply(
    lambda x: has_flag(x, "INVALID_DATE")
)

final_results_df["has_bank_account_mismatch"] = final_results_df["risk_flags"].apply(
    lambda x: has_flag(x, "BANK_ACCOUNT_MISMATCH")
)

final_results_df["has_duplicate"] = final_results_df["risk_flags"].apply(
    lambda x: has_flag(x, "DUPLICATE")
)

final_results_df["has_extraction_failed"] = final_results_df["extraction_status"].apply(
    lambda x: normalize_text_value(x, "").upper() == "FAILED"
)

# If original is_duplicate exists, combine it with risk flag duplicate
final_results_df["is_duplicate"] = (
    final_results_df["is_duplicate"] | final_results_df["has_duplicate"]
)

# Human approval flag should match final decision
final_results_df["needs_human_approval"] = final_results_df["final_decision"].apply(
    lambda x: x == "HUMAN_APPROVAL"
)

# Suspicious means not clean auto-post or has any risk
final_results_df["is_suspicious"] = (
    (final_results_df["final_decision"] != "AUTO_POST")
    | (final_results_df["risk_flags"] != "NONE")
    | (final_results_df["extraction_status"].str.upper() == "FAILED")
)

# Clean means AUTO_POST and no risk
final_results_df["is_clean"] = (
    (final_results_df["final_decision"] == "AUTO_POST")
    & (final_results_df["risk_flags"] == "NONE")
    & (final_results_df["extraction_status"].str.upper() != "FAILED")
)


# ------------------------------------------------------------
# 7. Improve denial reason if missing
# ------------------------------------------------------------

def build_denial_reason(row):
    current_reason = normalize_text_value(row.get("denial_reason", ""), "")

    if current_reason:
        return current_reason

    decision = row.get("final_decision", "")
    risk_flags = row.get("risk_flags", "NONE")

    if decision != "DENY":
        return ""

    if risk_flags == "NONE":
        return "Denied by business rule."

    return f"Denied because of detected risk: {risk_flags}"


final_results_df["denial_reason"] = final_results_df.apply(build_denial_reason, axis=1)


# ------------------------------------------------------------
# 8. Build clean, suspicious, and failed dataframes
# ------------------------------------------------------------

clean_invoices_df = final_results_df[final_results_df["is_clean"]].copy()

suspicious_invoices_df = final_results_df[final_results_df["is_suspicious"]].copy()

failed_files_df = final_results_df[
    final_results_df["extraction_status"].str.upper() == "FAILED"
].copy()


# ------------------------------------------------------------
# 9. Aggregate summary
# ------------------------------------------------------------

total_invoices = len(final_results_df)
clean_count = len(clean_invoices_df)
suspicious_count = len(suspicious_invoices_df)
failed_count = len(failed_files_df)

auto_post_count = int((final_results_df["final_decision"] == "AUTO_POST").sum())
human_approval_count = int((final_results_df["final_decision"] == "HUMAN_APPROVAL").sum())
deny_count = int((final_results_df["final_decision"] == "DENY").sum())

duplicate_count = int(final_results_df["is_duplicate"].sum())
amount_mismatch_count = int(final_results_df["has_amount_mismatch"].sum())
unregistered_vendor_count = int(final_results_df["has_unregistered_vendor"].sum())
invalid_date_count = int(final_results_df["has_invalid_date"].sum())
bank_mismatch_count = int(final_results_df["has_bank_account_mismatch"].sum())

image_invoice_count = int(
    final_results_df["file_type"].str.lower().isin(["jpg", "jpeg", "png"]).sum()
)

pdf_invoice_count = int(
    final_results_df["file_type"].str.lower().eq("pdf").sum()
)

# Handwritten detection may not exist yet; keep safe fallback
if "is_handwritten" in final_results_df.columns:
    handwritten_image_count = int(final_results_df["is_handwritten"].apply(lambda x: normalize_bool_value(x, False)).sum())
else:
    handwritten_image_count = 0

correct_invoice_count = clean_count

summary_data = {
    "total_invoices": total_invoices,
    "correct_invoices": correct_invoice_count,
    "clean_invoices": clean_count,
    "suspicious_invoices": suspicious_count,
    "failed_files": failed_count,
    "auto_post_count": auto_post_count,
    "human_approval_count": human_approval_count,
    "deny_count": deny_count,
    "duplicate_count": duplicate_count,
    "amount_mismatch_count": amount_mismatch_count,
    "unregistered_vendor_count": unregistered_vendor_count,
    "invalid_date_count": invalid_date_count,
    "bank_account_mismatch_count": bank_mismatch_count,
    "image_invoice_count": image_invoice_count,
    "pdf_invoice_count": pdf_invoice_count,
    "handwritten_image_invoice_count": handwritten_image_count,
}

summary_df = pd.DataFrame([summary_data])


# ------------------------------------------------------------
# 10. Save output files
# ------------------------------------------------------------

all_results_path = OUTPUT_DIR / "all_results.csv"
clean_path = OUTPUT_DIR / "clean_invoices.csv"
suspicious_path = OUTPUT_DIR / "suspicious_invoices.csv"
failed_path = OUTPUT_DIR / "failed_files.csv"
summary_path = OUTPUT_DIR / "aggregate_summary.csv"

final_results_df.to_csv(all_results_path, index=False, encoding="utf-8-sig")
clean_invoices_df.to_csv(clean_path, index=False, encoding="utf-8-sig")
suspicious_invoices_df.to_csv(suspicious_path, index=False, encoding="utf-8-sig")
failed_files_df.to_csv(failed_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# 11. Print judge-friendly summary
# ------------------------------------------------------------

print("\n========== AGGREGATE SUMMARY ==========")
print(f"Нийт invoice                  : {total_invoices}")
print(f"Зөв / clean invoice           : {clean_count}")
print(f"Сэжигтэй invoice              : {suspicious_count}")
print(f"Failed extraction             : {failed_count}")
print("---------------------------------------")
print(f"AUTO_POST                     : {auto_post_count}")
print(f"HUMAN_APPROVAL                : {human_approval_count}")
print(f"DENY                          : {deny_count}")
print("---------------------------------------")
print(f"Duplicate                     : {duplicate_count}")
print(f"Amount mismatch               : {amount_mismatch_count}")
print(f"Unregistered vendor           : {unregistered_vendor_count}")
print(f"Invalid date                  : {invalid_date_count}")
print(f"Bank account mismatch         : {bank_mismatch_count}")
print("---------------------------------------")
print(f"Image invoice                 : {image_invoice_count}")
print(f"PDF invoice                   : {pdf_invoice_count}")
print(f"Handwritten image invoice     : {handwritten_image_count}")
print("=======================================")


# ------------------------------------------------------------
# 12. Consistency checks
# ------------------------------------------------------------

print("\n========== CONSISTENCY CHECK ==========")

decision_total = auto_post_count + human_approval_count + deny_count

print("Total invoices               :", total_invoices)
print("Decision total               :", decision_total)
print("Clean + suspicious           :", clean_count + suspicious_count)

if decision_total == total_invoices:
    print("OK: decision count total matches invoice count")
else:
    print("ERROR: decision count total does not match invoice count")

if clean_count + suspicious_count == total_invoices:
    print("OK: clean + suspicious matches invoice count")
else:
    print("ERROR: clean + suspicious does not match invoice count")

if len(final_results_df) == len(results_df):
    print("OK: final_results_df keeps all batch rows")
else:
    print("ERROR: final_results_df row count changed unexpectedly")

print("=======================================")


# ------------------------------------------------------------
# 13. Output file check
# ------------------------------------------------------------

print("\n========== SAVED OUTPUT FILES ==========")

output_files = [
    all_results_path,
    clean_path,
    suspicious_path,
    failed_path,
    summary_path,
]

for path in output_files:
    if path.exists():
        print(f"OK: {path} | size={path.stat().st_size} bytes")
    else:
        print(f"ERROR: missing {path}")

print("========================================")


# ------------------------------------------------------------
# 14. Preview final output
# ------------------------------------------------------------

preview_cols = [
    "file_name",
    "file_type",
    "vendor_name",
    "category",
    "extraction_status",
    "final_decision",
    "risk_flags",
    "is_clean",
    "is_suspicious",
    "needs_human_approval",
]

existing_preview_cols = [col for col in preview_cols if col in final_results_df.columns]

print("\n========== FINAL RESULT PREVIEW ==========")
display(final_results_df[existing_preview_cols].head(20))

========== RESULT CONSOLIDATION START ==========
Initial results_df shape: (100, 39)

========== AGGREGATE SUMMARY ==========
Нийт invoice                  : 100
Зөв / clean invoice           : 68
Сэжигтэй invoice              : 32
Failed extraction             : 6
---------------------------------------
AUTO_POST                     : 68
HUMAN_APPROVAL                : 15
DENY                          : 17
---------------------------------------
Duplicate                     : 10
Amount mismatch               : 6
Unregistered vendor           : 0
Invalid date                  : 3
Bank account mismatch         : 8
---------------------------------------
Image invoice                 : 30
PDF invoice                   : 70
Handwritten image invoice     : 0

========== CONSISTENCY CHECK ==========
Total invoices               : 100
Decision total               : 100
Clean + suspicious           : 100
OK: decision count total matches invoice count
OK: clean + suspicious matches invoice co

,file_name,file_type,vendor_name,category,extraction_status,final_decision,risk_flags,is_clean,is_suspicious,needs_human_approval
0,invoice_001.pdf,pdf,Демо Компани-5,7,SUCCESS,HUMAN_APPROVAL,AMOUNT_MISMATCH,False,True,True
1,invoice_002.png,png,Демо Компани-8,5,SUCCESS,AUTO_POST,NONE,True,False,False
2,invoice_003.pdf,pdf,Демо Компани-10,5,SUCCESS,AUTO_POST,NONE,True,False,False
3,invoice_004.pdf,pdf,Демо Компани-1,8,SUCCESS,AUTO_POST,NONE,True,False,False
4,invoice_005.pdf,pdf,Демо Компани-4,10,SUCCESS,AUTO_POST,NONE,True,False,False
5,invoice_006.png,png,Демо Компани-1,8,SUCCESS,AUTO_POST,NONE,True,False,False
6,invoice_007.png,png,Демо Компани-5,7,SUCCESS,AUTO_POST,NONE,True,False,False
7,invoice_008.pdf,pdf,Демо Компани-1,8,SUCCESS,DENY,BANK_ACCOUNT_MISMATCH,False,True,False
8,invoice_009.png,png,Демо Компани-2,10,SUCCESS,AUTO_POST,NONE,True,False,False
9,invoice_010.png,png,Демо Компани-1,8,SUCCESS,DENY,BANK_ACCOUNT_MISMATCH,False,True,False


## 18. Mini Q&A Agent for Aggregate and Invoice Fact-checking

This section implements a lightweight Q&A agent over the processed invoice results.

The Q&A agent does not call an external LLM.  
Instead, it uses deterministic reasoning over `final_results_df` and `summary_df`.

It supports two main question types:

1. **Aggregate questions** — questions about the full dataset  
   Examples:
   - How many invoices are there?
   - How many invoices were denied?
   - How many invoices have amount mismatch?
   - How many invoices need human approval?

2. **Selected invoice fact-check questions** — questions about one specific invoice  
   Examples:
   - What is the final decision for invoice_001.pdf?
   - Is invoice_008.pdf duplicate?
   - What vendor does invoice_010.png have?
   - Does invoice_001.pdf need human approval?
   - Is the math calculation correct?

This rule-based Q&A layer improves consistency and avoids hallucination because all answers are derived directly from the final structured result table.

In [88]:
# ============================================================
# 18. Mini Q&A Agent for Aggregate and Invoice Fact-checking
# ============================================================

import re
import pandas as pd

print("========== MINI Q&A AGENT START ==========")

# ------------------------------------------------------------
# 1. Validate required dataframes
# ------------------------------------------------------------

if "final_results_df" not in globals():
    raise ValueError("final_results_df олдсонгүй. Эхлээд Section 17-г ажиллуулна уу.")

if "summary_df" not in globals():
    raise ValueError("summary_df олдсонгүй. Эхлээд Section 17-г ажиллуулна уу.")

if final_results_df.empty:
    raise ValueError("final_results_df хоосон байна.")

if summary_df.empty:
    raise ValueError("summary_df хоосон байна.")

print("final_results_df shape:", final_results_df.shape)
print("summary_df shape:", summary_df.shape)


# ------------------------------------------------------------
# 2. Text normalization helpers
# ------------------------------------------------------------

def q_normalize(text):
    """
    Question text-ийг keyword matching хийхэд тохиромжтой болгоно.
    """
    if text is None:
        return ""

    text = str(text).lower().strip()

    replacements = {
        "ё": "е",
        "ү": "ү",
        "ө": "ө",
        "_": " ",
        "-": " ",
        ".": " ",
        "?": " ",
        "!": " ",
        ",": " ",
        ":": " ",
        ";": " ",
        "(": " ",
        ")": " ",
        "[": " ",
        "]": " ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def contains_any(text, keywords):
    text = q_normalize(text)
    return any(q_normalize(k) in text for k in keywords)


def safe_get(row, col, default=""):
    if row is None:
        return default

    if col not in row.index:
        return default

    value = row[col]

    if pd.isna(value):
        return default

    return value


def bool_to_mn(value):
    if isinstance(value, str):
        value = value.strip().lower()
        if value in ["true", "yes", "1", "тийм"]:
            return "тийм"
        if value in ["false", "no", "0", "үгүй", "ugui"]:
            return "үгүй"

    return "тийм" if bool(value) else "үгүй"


def clean_value(value, default="тодорхойгүй"):
    if value is None:
        return default

    if isinstance(value, float) and pd.isna(value):
        return default

    text = str(value).strip()

    if text.lower() in ["", "nan", "none", "null"]:
        return default

    return text


# ------------------------------------------------------------
# 3. Summary value helper
# ------------------------------------------------------------

def get_summary_value(key, default=0):
    if key not in summary_df.columns:
        return default

    value = summary_df.iloc[0][key]

    try:
        return int(value)
    except Exception:
        return value


# ------------------------------------------------------------
# 4. Invoice detection from question
# ------------------------------------------------------------

def extract_invoice_number(question):
    """
    Question дотроос invoice number илрүүлнэ.

    Supported examples:
    - invoice_001.pdf
    - invoice 001
    - invoice 1
    - 001
    - 1-р invoice
    """
    q = str(question).lower()

    patterns = [
        r"invoice[_\-\s]*(\d{1,3})",
        r"(\d{1,3})\s*[-]?\s*р\s*invoice",
        r"(\d{1,3})\s*invoice",
    ]

    for pattern in patterns:
        match = re.search(pattern, q)
        if match:
            num = int(match.group(1))
            return f"{num:03d}"

    # If question is very short and only number-like
    short = q.strip()
    if re.fullmatch(r"\d{1,3}", short):
        return f"{int(short):03d}"

    return None


def find_invoice_row(question):
    """
    final_results_df дотроос invoice row олно.
    file_name дотор invoice number тааруулна.
    """
    invoice_num = extract_invoice_number(question)

    if invoice_num is None:
        return None, None

    pattern = f"invoice_{invoice_num}"

    matches = final_results_df[
        final_results_df["file_name"].astype(str).str.lower().str.contains(pattern, regex=False)
    ]

    if matches.empty:
        return invoice_num, None

    return invoice_num, matches.iloc[0]


# ------------------------------------------------------------
# 5. Aggregate Q&A handler
# ------------------------------------------------------------

def answer_aggregate_question(question):
    q = q_normalize(question)

    # Total invoices
    if contains_any(q, [
        "нийт хэдэн invoice",
        "нийт invoice",
        "total invoice",
        "all invoice",
        "хэдэн invoice байна",
        "how many invoices",
    ]):
        value = get_summary_value("total_invoices")
        return f"Нийт {value} invoice байна."

    # Correct / clean invoices
    if contains_any(q, [
        "зөв invoice",
        "correct invoice",
        "clean invoice",
        "алдаагүй invoice",
        "зөв хэдэн",
        "clean хэдэн",
    ]):
        value = get_summary_value("correct_invoices", get_summary_value("clean_invoices"))
        return f"Зөв буюу clean invoice: {value}."

    # Suspicious invoices
    if contains_any(q, [
        "сэжигтэй",
        "suspicious",
        "flagged",
        "эрсдэлтэй",
        "risk",
    ]):
        value = get_summary_value("suspicious_invoices")
        return f"Сэжигтэй invoice: {value}."

    # Duplicate invoices
    if contains_any(q, [
        "duplicate",
        "давхардсан",
        "давхардал",
        "давхар",
    ]):
        value = get_summary_value("duplicate_count")
        return f"Duplicate invoice: {value}."

    # Amount mismatch / math error
    if contains_any(q, [
        "математик",
        "тооцоолол",
        "дүн зөрүү",
        "amount mismatch",
        "math error",
        "calculation error",
        "mismatch amount",
        "буруу дүн",
    ]):
        value = get_summary_value("amount_mismatch_count")
        return f"Математик тооцоолол / amount mismatch алдаатай invoice: {value}."

    # Unregistered vendor
    if contains_any(q, [
        "бүртгэлгүй vendor",
        "unregistered vendor",
        "vendor бүртгэлгүй",
        "нийлүүлэгч бүртгэлгүй",
        "бүртгэлгүй нийлүүлэгч",
    ]):
        value = get_summary_value("unregistered_vendor_count")
        return f"Бүртгэлгүй vendor-той invoice: {value}."

    # Invalid date
    if contains_any(q, [
        "буруу огноо",
        "invalid date",
        "date error",
        "wrong date",
        "огнооны алдаа",
    ]):
        value = get_summary_value("invalid_date_count")
        return f"Буруу огноотой invoice: {value}."

    # Bank mismatch
    if contains_any(q, [
        "банк",
        "данс",
        "bank mismatch",
        "bank account mismatch",
        "account mismatch",
        "банкны мэдээллийн зөрүү",
        "дансны зөрүү",
    ]):
        value = get_summary_value("bank_account_mismatch_count")
        return f"Банкны мэдээлэл / дансны зөрүүтэй invoice: {value}."

    # Image invoices
    if contains_any(q, [
        "зураг хэлбэртэй",
        "image invoice",
        "jpg",
        "jpeg",
        "png",
        "image file",
    ]):
        value = get_summary_value("image_invoice_count")
        return f"Зураг хэлбэртэй invoice: {value}."

    # PDF invoices
    if contains_any(q, [
        "pdf invoice",
        "pdf хэдэн",
        "pdf файл",
    ]):
        value = get_summary_value("pdf_invoice_count")
        return f"PDF invoice: {value}."

    # Handwritten image invoices
    if contains_any(q, [
        "гар бичмэл",
        "handwritten",
        "hand writing",
        "гараар бичсэн",
    ]):
        value = get_summary_value("handwritten_image_invoice_count")
        return f"Гар бичмэлтэй зураг invoice: {value}."

    # Human approval
    if contains_any(q, [
        "human approval",
        "human approval авах",
        "хүн шалгах",
        "гараар шалгах",
        "manual review",
        "approval",
        "хүний баталгаажуулалт",
    ]):
        value = get_summary_value("human_approval_count")
        return f"HUMAN_APPROVAL шаардлагатай invoice: {value}."

    # DENY
    if contains_any(q, [
        "deny",
        "denied",
        "татгалзсан",
        "татгалзах",
        "reject",
        "rejected",
    ]):
        value = get_summary_value("deny_count")
        return f"DENY болсон invoice: {value}."

    # AUTO_POST
    if contains_any(q, [
        "auto post",
        "autopost",
        "auto_post",
        "автоматаар",
        "шууд батлах",
        "шууд оруулах",
    ]):
        value = get_summary_value("auto_post_count")
        return f"AUTO_POST болсон invoice: {value}."

    # Failed files
    if contains_any(q, [
        "failed",
        "амжилтгүй",
        "extraction failed",
        "уншиж чадаагүй",
        "алдаатай уншсан",
    ]):
        value = get_summary_value("failed_files")
        return f"Extraction failed invoice: {value}."

    return None


# ------------------------------------------------------------
# 6. Selected invoice fact-check handler
# ------------------------------------------------------------

def answer_invoice_fact_question(question):
    invoice_num, row = find_invoice_row(question)

    if invoice_num is None:
        return None

    if row is None:
        return f"invoice_{invoice_num} дугаартай invoice final result дотор олдсонгүй."

    q = q_normalize(question)

    file_name = clean_value(safe_get(row, "file_name"))
    vendor_name = clean_value(safe_get(row, "vendor_name"))
    category = clean_value(safe_get(row, "category"))
    due_date = clean_value(safe_get(row, "due_date"))
    decision = clean_value(safe_get(row, "final_decision"))
    risk_flags = clean_value(safe_get(row, "risk_flags"), "NONE")
    denial_reason = clean_value(safe_get(row, "denial_reason"), "")
    extraction_status = clean_value(safe_get(row, "extraction_status"), "UNKNOWN")

    is_duplicate = safe_get(row, "is_duplicate", False)
    vendor_registered = safe_get(row, "vendor_registered", False)
    bank_account_match = safe_get(row, "bank_account_match", False)
    math_correct = safe_get(row, "math_correct", False)
    needs_human_approval = safe_get(row, "needs_human_approval", False)

    # Final decision
    if contains_any(q, [
        "final decision",
        "шийдвэр",
        "decision",
        "ямар шийдвэр",
        "юу болсон",
        "route",
        "routing",
    ]):
        return (
            f"{file_name}-ийн final decision: {decision}. "
            f"Risk flags: {risk_flags}."
        )

    # Duplicate
    if contains_any(q, [
        "duplicate",
        "давхардсан",
        "давхардал",
        "давхар",
    ]):
        return f"{file_name} duplicate мөн үү? {bool_to_mn(is_duplicate)}."

    # Vendor name
    if contains_any(q, [
        "vendor нэр",
        "vendor",
        "нийлүүлэгч",
        "компани",
        "байгууллага",
    ]):
        return f"{file_name}-ийн vendor: {vendor_name}."

    # Category
    if contains_any(q, [
        "category",
        "ангилал",
        "төрөл",
        "ямар category",
    ]):
        return f"{file_name}-ийн category: {category}."

    # Due date
    if contains_any(q, [
        "due date",
        "төлөх огноо",
        "дуусах огноо",
        "хугацаа",
        "due",
    ]):
        return f"{file_name}-ийн due date: {due_date}."

    # Bank account registered / matched
    if contains_any(q, [
        "bank account бүртгэлтэй",
        "bank registered",
        "account registered",
        "данс бүртгэлтэй",
        "банкны данс",
        "bank account",
        "данс",
    ]):
        answer = "тийм" if bool(bank_account_match) else "үгүй"
        return (
            f"{file_name}-ийн bank account database-тэй таарч байна уу? {answer}. "
            f"bank_account_match = {bool(bank_account_match)}."
        )

    # Why denied
    if contains_any(q, [
        "яагаад deny",
        "why deny",
        "why denied",
        "denial reason",
        "deny reason",
        "яагаад татгалзсан",
        "татгалзсан шалтгаан",
    ]):
        if decision != "DENY":
            return (
                f"{file_name} DENY болоогүй. "
                f"Final decision: {decision}. Risk flags: {risk_flags}."
            )

        if denial_reason:
            return f"{file_name} DENY болсон шалтгаан: {denial_reason}"

        return f"{file_name} DENY болсон шалтгаан: {risk_flags}."

    # Error types / risk flags
    if contains_any(q, [
        "алдаа",
        "error",
        "risk flag",
        "risk_flags",
        "ямар төрлийн алдаа",
        "зөрчил",
        "issue",
    ]):
        return f"{file_name}-д илэрсэн error / risk flags: {risk_flags}."
        
    # Why human approval
    if contains_any(q, [
        "яагаад human",
        "яагаад approval",
        "why human",
        "why approval",
        "manual review reason",
        "яагаад хүн шалгах",
        "яагаад гараар шалгах",
    ]):
        if decision != "HUMAN_APPROVAL":
            return (
                f"{file_name} HUMAN_APPROVAL биш. "
                f"Final decision: {decision}. Risk flags: {risk_flags}."
            )

        return (
            f"{file_name} HUMAN_APPROVAL болсон шалтгаан: "
            f"{risk_flags}. Энэ төрлийн зөрчлийг хүний шалгалтаар баталгаажуулна."
        )

    # Human approval
    if contains_any(q, [
        "human approval",
        "хүн шалгах",
        "гараар шалгах",
        "manual review",
        "approval",
        "human",
    ]):
        return f"{file_name} HUMAN_APPROVAL авах ёстой юу? {bool_to_mn(needs_human_approval)}."

    # Math correct
    if contains_any(q, [
        "математик",
        "тооцоолол",
        "math",
        "calculation",
        "дүн зөв",
        "math correct",
    ]):
        return f"{file_name}-ийн математик тооцоолол зөв үү? {bool_to_mn(math_correct)}."

    # Vendor registered
    if contains_any(q, [
        "vendor бүртгэлтэй",
        "registered vendor",
        "vendor registered",
        "нийлүүлэгч бүртгэлтэй",
    ]):
        return f"{file_name}-ийн vendor database-д бүртгэлтэй юу? {bool_to_mn(vendor_registered)}."

    # Extraction status
    if contains_any(q, [
        "status",
        "extraction",
        "уншсан",
        "амжилттай уншсан",
        "ocr",
    ]):
        return f"{file_name}-ийн extraction status: {extraction_status}."

    # Default selected invoice summary
    return (
        f"{file_name} summary: "
        f"vendor={vendor_name}, category={category}, due_date={due_date}, "
        f"decision={decision}, risk_flags={risk_flags}."
    )


# ------------------------------------------------------------
# 7. Unified answer function
# ------------------------------------------------------------

def answer_question(question):
    """
    Main Q&A function.
    First checks whether the question is about a selected invoice.
    If not, it answers aggregate questions.
    """
    if question is None or str(question).strip() == "":
        return "Асуулт хоосон байна. Invoice summary эсвэл invoice дугаартай fact-check асуулт асууна уу."

    # If invoice number exists, try selected invoice fact-check first
    invoice_answer = answer_invoice_fact_question(question)
    if invoice_answer is not None:
        return invoice_answer

    # Otherwise aggregate Q&A
    aggregate_answer = answer_aggregate_question(question)
    if aggregate_answer is not None:
        return aggregate_answer

    return (
        "Энэ асуултад одоогоор deterministic Q&A agent шууд хариулж чадсангүй. "
        "Нийт count, suspicious/duplicate/error count, эсвэл invoice_001.pdf гэх мэт тодорхой invoice-ийн fact-check асуулт асууна уу."
    )


# ------------------------------------------------------------
# 8. Test questions
# ------------------------------------------------------------

test_questions = [
    "Нийт хэдэн invoice байна вэ?",
    "Хэдэн invoice зөв invoice вэ?",
    "Хэдэн invoice сэжигтэй вэ?",
    "Хэдэн invoice duplicate вэ?",
    "Хэдэн invoice математик тооцооллын алдаатай вэ?",
    "Хэдэн invoice бүртгэлгүй vendor-той вэ?",
    "Хэдэн invoice буруу огноотой вэ?",
    "Хэдэн invoice банкны мэдээллийн зөрүүтэй вэ?",
    "Хэдэн invoice зураг хэлбэртэй вэ?",
    "Хэдэн invoice гар бичмэлтэй зураг вэ?",
    "Хэдэн invoice HUMAN_APPROVAL авах ёстой вэ?",
    "Хэдэн invoice DENY болох ёстой вэ?",
    "invoice_001.pdf final decision юу вэ?",
    "invoice_001.pdf duplicate мөн үү?",
    "invoice_001.pdf vendor-ийн нэр юу вэ?",
    "invoice_001.pdf ямар category-д ангилагдсан бэ?",
    "invoice_001.pdf due date хэд вэ?",
    "invoice_001.pdf bank account бүртгэлтэй эсэх?",
    "invoice_001.pdf яагаад deny болсон бэ?",
    "invoice_001.pdf ямар төрлийн алдаа илэрсэн бэ?",
    "invoice_001.pdf human approval авах ёстой юу?",
    "invoice_001.pdf математик тооцоолол зөв үү?",
]

print("\n========== MINI Q&A TEST ==========")

for q in test_questions:
    print("\nQ:", q)
    print("A:", answer_question(q))

print("\n========== MINI Q&A AGENT READY ==========")

========== MINI Q&A AGENT START ==========
final_results_df shape: (100, 58)
summary_df shape: (1, 16)

========== MINI Q&A TEST ==========

Q: Нийт хэдэн invoice байна вэ?
A: Нийт 100 invoice байна.

Q: Хэдэн invoice зөв invoice вэ?
A: Зөв буюу clean invoice: 68.

Q: Хэдэн invoice сэжигтэй вэ?
A: Сэжигтэй invoice: 32.

Q: Хэдэн invoice duplicate вэ?
A: Duplicate invoice: 10.

Q: Хэдэн invoice математик тооцооллын алдаатай вэ?
A: Математик тооцоолол / amount mismatch алдаатай invoice: 6.

Q: Хэдэн invoice бүртгэлгүй vendor-той вэ?
A: Бүртгэлгүй vendor-той invoice: 0.

Q: Хэдэн invoice буруу огноотой вэ?
A: Буруу огноотой invoice: 3.

Q: Хэдэн invoice банкны мэдээллийн зөрүүтэй вэ?
A: Банкны мэдээлэл / дансны зөрүүтэй invoice: 8.

Q: Хэдэн invoice зураг хэлбэртэй вэ?
A: Зураг хэлбэртэй invoice: 30.

Q: Хэдэн invoice гар бичмэлтэй зураг вэ?
A: Гар бичмэлтэй зураг invoice: 0.

Q: Хэдэн invoice HUMAN_APPROVAL авах ёстой вэ?
A: HUMAN_APPROVAL шаардлагатай invoice: 15.

Q: Хэдэн invoice DENY

## 19. Final Output Verification

This section verifies that the final notebook outputs are complete, consistent, and ready for submission.

It checks:

- Whether all required CSV files exist in `/kaggle/working/`
- Whether `all_results.csv` contains all processed invoices
- Whether clean, suspicious, failed, and summary outputs are consistent
- Whether final decision counts match the total invoice count
- Whether aggregate Q&A values match the exported summary

This final verification helps confirm that the notebook is reproducible and judge-friendly.

In [90]:
# ============================================================
# 19. Final Output Verification
# ============================================================

from pathlib import Path
import pandas as pd

print("========== FINAL OUTPUT VERIFICATION START ==========")

OUTPUT_DIR = Path("/kaggle/working")

required_output_files = {
    "all_results": OUTPUT_DIR / "all_results.csv",
    "clean_invoices": OUTPUT_DIR / "clean_invoices.csv",
    "suspicious_invoices": OUTPUT_DIR / "suspicious_invoices.csv",
    "failed_files": OUTPUT_DIR / "failed_files.csv",
    "aggregate_summary": OUTPUT_DIR / "aggregate_summary.csv",
}

# ------------------------------------------------------------
# 1. Check file existence
# ------------------------------------------------------------

print("\n========== REQUIRED OUTPUT FILE CHECK ==========")

missing_files = []

for name, path in required_output_files.items():
    if path.exists():
        print(f"OK: {name} -> {path} | size={path.stat().st_size} bytes")
    else:
        print(f"ERROR: missing {name} -> {path}")
        missing_files.append(name)

if missing_files:
    raise FileNotFoundError(f"Missing required output files: {missing_files}")

print("All required output files exist.")


# ------------------------------------------------------------
# 2. Load exported files
# ------------------------------------------------------------

exported_all_df = pd.read_csv(required_output_files["all_results"])
exported_clean_df = pd.read_csv(required_output_files["clean_invoices"])
exported_suspicious_df = pd.read_csv(required_output_files["suspicious_invoices"])
exported_failed_df = pd.read_csv(required_output_files["failed_files"])
exported_summary_df = pd.read_csv(required_output_files["aggregate_summary"])

print("\n========== EXPORTED FILE SHAPES ==========")
print("all_results.csv        :", exported_all_df.shape)
print("clean_invoices.csv     :", exported_clean_df.shape)
print("suspicious_invoices.csv:", exported_suspicious_df.shape)
print("failed_files.csv       :", exported_failed_df.shape)
print("aggregate_summary.csv  :", exported_summary_df.shape)


# ------------------------------------------------------------
# 3. Required column check
# ------------------------------------------------------------

required_columns = [
    "file_name",
    "file_type",
    "vendor_name",
    "category",
    "extraction_status",
    "final_decision",
    "risk_flags",
    "is_clean",
    "is_suspicious",
    "needs_human_approval",
]

print("\n========== REQUIRED COLUMN CHECK ==========")

missing_columns = [
    col for col in required_columns
    if col not in exported_all_df.columns
]

if missing_columns:
    print("ERROR: Missing columns:", missing_columns)
else:
    print("OK: all required columns exist in all_results.csv")


# ------------------------------------------------------------
# 4. Count consistency check
# ------------------------------------------------------------

print("\n========== COUNT CONSISTENCY CHECK ==========")

total_rows = len(exported_all_df)
clean_rows = len(exported_clean_df)
suspicious_rows = len(exported_suspicious_df)
failed_rows = len(exported_failed_df)

auto_post_count = int((exported_all_df["final_decision"] == "AUTO_POST").sum())
human_approval_count = int((exported_all_df["final_decision"] == "HUMAN_APPROVAL").sum())
deny_count = int((exported_all_df["final_decision"] == "DENY").sum())

decision_total = auto_post_count + human_approval_count + deny_count
clean_plus_suspicious = clean_rows + suspicious_rows

print("Total rows             :", total_rows)
print("Clean rows             :", clean_rows)
print("Suspicious rows        :", suspicious_rows)
print("Failed rows            :", failed_rows)
print("---------------------------------------")
print("AUTO_POST              :", auto_post_count)
print("HUMAN_APPROVAL         :", human_approval_count)
print("DENY                   :", deny_count)
print("Decision total         :", decision_total)
print("---------------------------------------")
print("Clean + suspicious     :", clean_plus_suspicious)

if decision_total == total_rows:
    print("OK: decision total matches total rows")
else:
    print("ERROR: decision total does not match total rows")

if clean_plus_suspicious == total_rows:
    print("OK: clean + suspicious matches total rows")
else:
    print("ERROR: clean + suspicious does not match total rows")


# ------------------------------------------------------------
# 5. Summary consistency check
# ------------------------------------------------------------

print("\n========== SUMMARY CONSISTENCY CHECK ==========")

summary_row = exported_summary_df.iloc[0].to_dict()

checks = {
    "total_invoices": total_rows,
    "clean_invoices": clean_rows,
    "suspicious_invoices": suspicious_rows,
    "failed_files": failed_rows,
    "auto_post_count": auto_post_count,
    "human_approval_count": human_approval_count,
    "deny_count": deny_count,
}

summary_errors = []

for key, actual_value in checks.items():
    expected_value = int(summary_row.get(key, -1))

    if expected_value == actual_value:
        print(f"OK: {key} = {actual_value}")
    else:
        print(f"ERROR: {key} summary={expected_value}, actual={actual_value}")
        summary_errors.append(key)

if not summary_errors:
    print("OK: aggregate_summary.csv matches exported result files")
else:
    print("ERROR: summary mismatch found:", summary_errors)


# ------------------------------------------------------------
# 6. Risk flag consistency check
# ------------------------------------------------------------

print("\n========== RISK FLAG CHECK ==========")

if "risk_flags" in exported_all_df.columns:
    unknown_risk_rows = exported_all_df[
        exported_all_df["risk_flags"].isna()
        | (exported_all_df["risk_flags"].astype(str).str.strip() == "")
    ]

    if len(unknown_risk_rows) == 0:
        print("OK: no empty risk_flags")
    else:
        print("WARNING: empty risk_flags rows:", len(unknown_risk_rows))

if "final_decision" in exported_all_df.columns:
    invalid_decisions = exported_all_df[
        ~exported_all_df["final_decision"].isin(["AUTO_POST", "HUMAN_APPROVAL", "DENY"])
    ]

    if len(invalid_decisions) == 0:
        print("OK: all final_decision values are valid")
    else:
        print("ERROR: invalid final_decision rows:", len(invalid_decisions))
        display(invalid_decisions[["file_name", "final_decision", "risk_flags"]].head(10))


# ------------------------------------------------------------
# 7. Final submission summary
# ------------------------------------------------------------

print("\n========== FINAL SUBMISSION SUMMARY ==========")
print(f"Total invoices processed        : {total_rows}")
print(f"Clean / correct invoices        : {clean_rows}")
print(f"Suspicious invoices             : {suspicious_rows}")
print(f"Failed extraction invoices      : {failed_rows}")
print("---------------------------------------------")
print(f"AUTO_POST decisions             : {auto_post_count}")
print(f"HUMAN_APPROVAL decisions        : {human_approval_count}")
print(f"DENY decisions                  : {deny_count}")
print("---------------------------------------------")

if (
    len(missing_files) == 0
    and len(missing_columns) == 0
    and decision_total == total_rows
    and clean_plus_suspicious == total_rows
    and len(summary_errors) == 0
):
    print("FINAL STATUS: READY FOR SUBMISSION")
else:
    print("FINAL STATUS: NEEDS REVIEW")

print("==============================================")

========== FINAL OUTPUT VERIFICATION START ==========

========== REQUIRED OUTPUT FILE CHECK ==========
OK: all_results -> /kaggle/working/all_results.csv | size=139491 bytes
OK: clean_invoices -> /kaggle/working/clean_invoices.csv | size=96605 bytes
OK: suspicious_invoices -> /kaggle/working/suspicious_invoices.csv | size=43723 bytes
OK: failed_files -> /kaggle/working/failed_files.csv | size=6336 bytes
OK: aggregate_summary -> /kaggle/working/aggregate_summary.csv | size=355 bytes
All required output files exist.

========== EXPORTED FILE SHAPES ==========
all_results.csv        : (100, 58)
clean_invoices.csv     : (68, 58)
suspicious_invoices.csv: (32, 58)
failed_files.csv       : (6, 58)
aggregate_summary.csv  : (1, 16)

========== REQUIRED COLUMN CHECK ==========
OK: all required columns exist in all_results.csv

========== COUNT CONSISTENCY CHECK ==========
Total rows             : 100
Clean rows             : 68
Suspicious rows        : 32
Failed rows            : 6
------------

## 20. Optional Gradio Interface

Энэ cell нь demo хийхэд ашиглаж болох interface үүсгэнэ. Kaggle дээр public URL ажиллах эсэх нь runtime орчноос шалтгаална. Notebook-ийн үндсэн pipeline interface-гүй ч бүрэн ажиллана.

In [ ]:
ENABLE_GRADIO_DEMO = True  # Interface ажиллуулах бол True болгоно.

if ENABLE_GRADIO_DEMO:
    import gradio as gr

    chatbot_df = results_df.copy()

    def gradio_ask(question):
        return answer_question(question, chatbot_df)

    def gradio_process_file(file):
        if file is None:
            return 'Файл upload хийгээгүй байна.', None
        path = Path(file.name)
        result = process_single_invoice(path)
        df = pd.DataFrame([result])
        return result.get('explanation', ''), df

    with gr.Blocks(title='Invoice Automation AI Agent') as demo:
        gr.Markdown('# Invoice Automation AI Agent')
        gr.Markdown('PDF/JPG/PNG invoice upload хийж, validation result болон chatbot Q&A ашиглана.')

        with gr.Tab('Single Invoice Demo'):
            file_input = gr.File(label='Upload invoice file')
            process_btn = gr.Button('Process Invoice')
            explanation_output = gr.Textbox(label='Explanation')
            table_output = gr.Dataframe(label='Result')
            process_btn.click(gradio_process_file, inputs=file_input, outputs=[explanation_output, table_output])

        with gr.Tab('Chatbot Q&A'):
            question = gr.Textbox(label='Ask a question', placeholder='Хэдэн invoice DENY болсон бэ?')
            answer = gr.Textbox(label='Answer')
            ask_btn = gr.Button('Ask')
            ask_btn.click(gradio_ask, inputs=question, outputs=answer)

    demo.launch(share=True)
else:
    print('Gradio demo disabled. ENABLE_GRADIO_DEMO=True болгож ажиллуулж болно.')

In [ ]:
# ============================================================
# 20. Final Submission File Check
# ============================================================

from pathlib import Path

for filename in [
    "final_results.csv",
    "failed_files.csv",
    "aggregate_answers.csv",
    "auto_post_invoices.csv",
    "human_approval_invoices.csv",
    "denied_invoices.csv",
]:
    path = Path("/kaggle/working") / filename
    print(filename, "exists:", path.exists(), "size:", path.stat().st_size if path.exists() else 0)

print("\n========== FINAL COUNTS ==========")
print("Total:", len(final_df))
print("AUTO_POST:", (final_df["final_decision"].astype(str) == "AUTO_POST").sum())
print("HUMAN_APPROVAL:", (final_df["final_decision"].astype(str) == "HUMAN_APPROVAL").sum())
print("DENY:", (final_df["final_decision"].astype(str) == "DENY").sum())
print("SUCCESS:", (final_df["processing_status"].astype(str).str.upper() == "SUCCESS").sum())
print("FAILED:", (final_df["processing_status"].astype(str).str.upper() != "SUCCESS").sum())
print("=================================")

## 21. Limitations and Future Improvements

**Limitations:**

- Extraction accuracy depends on invoice image quality and Vision model response.
- Bank account validation works only if master database contains bank account columns.
- Category classification uses historical matching and keyword rules, not a trained classifier.
- Chatbot Q&A is analytical and rule-based; it is designed for reliable numeric answers.

**Future improvements:**

- Add OCR + layout parser fallback for API-free mode.
- Improve duplicate detection using embedding similarity.
- Add confidence scoring for extracted fields.
- Expand chatbot with LLM-based natural language interpretation.
- Deploy full Streamlit/Gradio web app.

In [ ]:
print("results_df exists:", "results_df" in globals())